# Model ENSO predictclim

August 2023

Caroline Juang, c.juang@columbia.edu

**The purpose of this file is to output the counterfactual climate predictions, but it only outputs for climate variables that are relevant to modeling burned area.**

**Inputs:**
* SST gradient (1982-present) - build it following the README
* trained models `Model_ENSOclim_Akaike` (SST gradient -> climate)
* trained models `Model_Akaike` (climate -> burned area)

**Variables**
* `sstpred` generally refers to the detrended SST gradient, which is the detrended-SST gradient scenario put into the climate->burned area model.
* `climpred` generally refers to the observed SST gradient, put into the climate->burned area model.

**Outputs:**
* Predicted climate by season and organized by current-year and prior-year (1983-present) (but only for the relevant variables to the climate->burned area model).

The model is based on the outputs of `Model_ENSOclim_Akaike` and `Model_Akaike`.

Data source:
* NOAA sea surface temperature, https://psl.noaa.gov/data/gridded/data.noaa.ersst.v5.html

save and load models

Machine Learning Mastery: https://machinelearningmastery.com/save-load-machine-learning-models-python-scikit-learn/

In [1]:
# import
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
import joblib

In [2]:
# customize seasons for climate variables

# customize number of rolling periods
ant_years = 1 # antecedent years to include (2 antecedent years + current year)
ant_season = 3 # n+1 of months to include in each period (e.g. input 2 would mean 3 months)

firstyear = 1984 # first year of data
finalyear = 2022 # final year of data (should be same as burned area)
time_length = int(finalyear-firstyear+1) # get length of timeseries
# translate years into dates
firsttime = str(firstyear)+'-01-01'
finaltime = str(finalyear)+'-12-31'
years = np.arange(firstyear, finalyear+1)

# importing data string
# Read the climate index from the .txt file
with open('0_climindname.txt', 'r') as f:
    climindname = f.read().strip()
print(climindname +' will be used for the SST gradient')
climindnameDT = 'DeTrend_'+climindname
climfilename = 'DeTrendClimObs_'+climindname # sst-predicted BA

directory = 'your_data_folder' # customize this
#data_string = 'data\\'
data_string = 'data//'
#model_string = 'model\\'+climindname+'\\'
model_string = 'model//'+climindname+'//'
#predict_string = 'predicted\\'+climindname+'\\'
predict_string = 'predicted//'+climindname+'//'

# folder for saving figures
figfolder = 'your_figures_folder' # customize this

# create strings of the types of forests, remove #11
province_names = ['1: American Semi-Desert and Desert Province',
                  '2: Arizona-New Mexico Mountains Semi-Desert-Open Woodland-Coniferous Forest-Alpine Meadow Province',
                  '3: Black Hills Coniferous Forest Province',
                  '4: California Coastal Chapparral Forest and Shrub Province',
                  '5: California Coastal Range Open Woodland-Shrub-Coniferous Forest-Meadow Province',
                  '6: California Coastal Steppe-Mixed Forest-Redwood Forest Province',
                  '7: California Dry Steppe Province',
                  '8: Cascade Mixed Forest-Coniferous Forest-Alpine Meadow Province',
                  '9: Chihuahuan Semi-Desert Province',
                  '10: Colorado Plateau Semi-Desert Province',
                  '12: Great Plains-Palouse Dry Steppe Province',
                  '13: Intermountain Semi-Desert Province',
                  '14: Intermountain Semi-Desert and Desert Province',
                  '15: Middle Rocky Mountain Steppe-Coniferous Forest-Alpine Meadow Province',
                  '16: Nevada-Utah Mountains-Semi-Desert-Coniferous Forest-Alpine Meadow Province',
                  '17: Northern Rocky Mountain Forest-Steppe-Coniferous Forest-Alpine Meadow Province',
                  '18: Pacific Lowland Mixed Forest Province',
                  '19: Sierran Steppe-Mixed Forest-Coniferous Forest-Alpine Meadow Province',
                  '20: Southern Rocky Mountain Steppe-Open Woodland-Coniferous Forest-Alpine Meadow Province',
                  '21: Southwest Plateau and Plains Dry Steppe and Shrub Province']

province_num = [item for item in range(len(province_names)+1+1)]
province_num.remove(11) # remove empty ecoregion - no overlap btwn westUS map and ecoregion
dfnames = ['allwestUS', 'ecoprov1', 'ecoprov2', 'ecoprov3', 'ecoprov4', 'ecoprov5', 
           'ecoprov6', 'ecoprov7', 'ecoprov8', 'ecoprov9', 'ecoprov10', 
           'ecoprov12','ecoprov13','ecoprov14','ecoprov15',
           'ecoprov16','ecoprov17','ecoprov18','ecoprov19','ecoprov20','ecoprov21']

patch125-155_nino3-34 will be used for the SST gradient


In [3]:
# average climate variables, within a selected season

def annual_seasonAvg(data, firstmonth, finalmonth):
    """
    This intakes an array of current ecoregion's climate variable of monthly averages (data), 
    Output: an array of yearly averages of the ecoregion climate variable, in the
    seasons specified (firstmonth, finalmonth).
    Requirements: time = an xarray timeseries of months in datetime format.
    """
    withinyear = (time['time.year']>= years[0]) & (time['time.year'] <= years[-1])
    withinseason = (time['time.month'] >= firstmonth) & (time['time.month'] <= finalmonth)
    thistime = time[withinseason & withinyear] # cut time

    # create pd dataframe based on data, for time resampling
    thisdf = pd.DataFrame({'time': pd.to_datetime(thistime.values), 'clim':data[withinyear & withinseason]}).set_index('time')
    thisdf = thisdf.resample('Y').mean().reset_index()
    return thisdf.clim.values

## Import observed SST gradient data
from `Data_CreateENSOIndex` and `Data_CreateModelData`

In [4]:
# import observed SST gradient
filename = data_string + 'sstgrad_seasons_' + climindname+'_82_y.txt'
climind82_seasons = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# import de-trended SST gradient
filename = data_string + 'sstgrad_seasons_' + climindnameDT+'_82_y.txt'
climind82_seasonsDT = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# import observed SST gradient (with extra year, for comparison)
filename = data_string + 'sstgrad_seasons_' + climindname+'_81_y.txt'
climind81_seasons = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# IMPORT gSST AVERAGE (average of concurrent-season and prior-season)
# import observed SST gradient
filename = data_string + 'sstgrad_seasons_' + climindname+'_82_y_avgcurr-prior.txt'
climind82_seasonsavg = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# import de-trended SST gradient
filename = data_string + 'sstgrad_seasons_' + climindnameDT+'_82_y_avgcurr-prior.txt'
climind82_seasonsDTavg = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# put the dataframes together
climind82_seasons = pd.concat([climind82_seasons, climind82_seasonsavg], axis=1)
climind82_seasonsDT = pd.concat([climind82_seasonsDT, climind82_seasonsDTavg], axis=1)

imported data//sstgrad_seasons_patch125-155_nino3-34_82_y.txt
imported data//sstgrad_seasons_DeTrend_patch125-155_nino3-34_82_y.txt
imported data//sstgrad_seasons_patch125-155_nino3-34_81_y.txt
imported data//sstgrad_seasons_patch125-155_nino3-34_82_y_avgcurr-prior.txt
imported data//sstgrad_seasons_DeTrend_patch125-155_nino3-34_82_y_avgcurr-prior.txt


## Import observed climate data
from `Data_CreateModelData`

In [5]:
# import OBSERVED climate data to feed into models
# climate is predicted using SST, but this is the climate observed data to check

dfframesfor = {}
dfframesnon = {}

climind = pd.read_csv(data_string + 'gradient_'+climindname+'.txt', header=None, sep=",", skiprows=[0]).set_index(0)
for iecoreg in np.arange(len(province_num)):
    filename = data_string + 'climate_ecoprovinces_'
    print(dfnames[iecoreg])
    dfframesfor[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_for.txt').set_index('Unnamed: 0')
    dfframesnon[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_non.txt').set_index('Unnamed: 0')

allwestUS
ecoprov1
ecoprov2
ecoprov3
ecoprov4
ecoprov5
ecoprov6
ecoprov7
ecoprov8
ecoprov9
ecoprov10
ecoprov12
ecoprov13
ecoprov14
ecoprov15
ecoprov16
ecoprov17
ecoprov18
ecoprov19
ecoprov20
ecoprov21


## Import model outputs
from `Model_ENSOclim_Akaike` and `Model_Akaike`

In [6]:
# get the model input variable names from the txt files
# sst gradient to predict climate variables

with open(model_string + "modeloutput_sst_all_all.txt", "r") as f:
    inputsall = [line.strip() for line in f]
f.close()
with open(model_string + "modeloutput_sst_for.txt", "r") as f:
    inputsfor = [line.strip() for line in f]
f.close()
with open(model_string + "modeloutput_sst_non.txt", "r") as f:
    inputsnon = [line.strip() for line in f]
f.close()

# get indices of where each ecoregion's outputs begin
iinputsall = [i for i, e in 
               enumerate(inputsall) if "+++" in e]
# get ecoregions so iteration is not manual
iinputsfor = [i for i, e in 
               enumerate(inputsfor) if "+++" in e]
# get ecoregions so iteration is not manual
iinputsnon = [i for i, e in 
               enumerate(inputsnon) if "+++" in e]
# add in last index
iinputsall.append(len(inputsall)+1)
iinputsfor.append(len(inputsfor)+1)
iinputsnon.append(len(inputsnon)+1)

In [7]:
# get the model input variable names from the txt files
# climate variables to predict burned area

with open(model_string + "modeloutput_burnarea_for.txt", "r") as f:
    inputsfor_clim = [line.strip() for line in f]
f.close()
with open(model_string + "modeloutput_burnarea_non.txt", "r") as f:
    inputsnon_clim = [line.strip() for line in f]
f.close()

# get indices of where each ecoregion's outputs begin
# get ecoregions so iteration is not manual
iinputsfor_clim = [i for i, e in 
               enumerate(inputsfor_clim) if "+++" in e]
# get ecoregions so iteration is not manual
iinputsnon_clim = [i for i, e in 
               enumerate(inputsnon_clim) if "+++" in e]
# add in last index
iinputsfor_clim.append(len(inputsfor_clim)+1)
iinputsnon_clim.append(len(inputsnon_clim)+1)

# Setup for exporting burned area predictions
Predicted climate
* SST_obs-predicted climate
* SST_neg-predicted climate
* difference between climate_sstobs and climate_sstneg, called climate_preddiff
* observed climate with climate_preddiff subtracted from it, called climate_sstdiff

Predicted burned area
* SST-predicted burned area (observed SST, goes through SST-clim and clim-BA models)
* climate-predicted burned area (observed climate, goes through clim-BA model)

In [8]:
# a bunch of dictionaries for storage
# predicted climate from observed SST
dfsstpredclimall_obsSST = {}
dfsstpredclimfor_obsSST = {}
dfsstpredclimnon_obsSST = {}

# predicted climate from negative-trended SST
dfsstpredclimall_negSST = {}
dfsstpredclimfor_negSST = {}
dfsstpredclimnon_negSST = {}

# difference in predicted climate scenarios, climate_preddiff
dfsstpredclimall_preddiff = {}
dfsstpredclimfor_preddiff = {}
dfsstpredclimnon_preddiff = {}

# observed climate minus climate_preddiff
dfsstpredclimall_sstdiff = {}
dfsstpredclimfor_sstdiff = {}
dfsstpredclimnon_sstdiff = {}

# scripts

In [9]:
# get the SST inputs needed
def modelinputsst_string(inputlist, varname, istart, iend):
    """
    Requirements: 
    inputlist = list of the Model_ENSOclim_Akaike model results (locally defined)
    ithisecoreg = the ecoregion name (globally defined)
    ithisecoregend = the next ecoregion name 
        in the list (globally defined)
    varname = the climate variable name (locally defined)
    """
    # narrow inputs list to ecoregion
    modellist = inputlist[istart:iend+1]
    # get all PREDICTING (where inputs start)
    iinputs = [i for i, e in 
               enumerate(modellist) if "PREDICTING" in e]
    # extract model inputs
    istart = modellist.index('PREDICTING ' + varname)
    if len(iinputs)>((iinputs.index(istart))+1): # 
        iend = iinputs[(iinputs.index(istart))+1]
        modelinputs = modellist[istart+1:iend] # range in list
    else:
        modelinputs = modellist[istart+1:] # last predictor to end of list
    return modelinputs

# SST-observed -> climate

## Ecoregion SST->climate

The goal is to predict every single climate using observed and detrended SST gradient, then get difference between the SST->climate model predictions, and **output all detrended-scenario climates.**

In [10]:
# set up storage
dfframesforDT = {}
dfframesnonDT = {}

# setup for storing the multidimensional climate data
# shape is (ecoprovinces, years, climate variables)
climbyecoregfor = [] # observed SST into SST-clim model
climbyecoregnon = []
climbyecoregforDT = [] # detrended SST into SST-clim model
climbyecoregnonDT = []

# get list of variables
climallnames = dfframesfor['allwestUS'].columns.values

## SST gradient is observed, SST->climate

In [11]:
# iterate through each FOREST ecoregion
# calculate the climate as predicted by the SST gradient

SSTgradient = climind82_seasons

# separate concurrent and previous-year variables
climy0names = [s for s in climallnames if "y0" in s]
climy1names = [s for s in climallnames if "y-1" in s]


for ecoregname in dfnames:
    print(ecoregname)
    landname = 'for'
    landtypeinput = inputsfor
    landtypeiinput = iinputsfor

    ithisecoreg = landtypeinput.index('+++'+ecoregname)
    ithisecoregend = landtypeiinput[landtypeiinput.index(ithisecoreg)+1]-1
    # storage
    climmodeldict = {} # the loaded model
    climpred = [] # predicted climate variable values
    climnames = [] # name of climate variable
    
    # open models
    if len(climy0names)>0:
        for thisname in climy0names:
            #print(thisname)
            filename = model_string + 'model_'+landname+'_'+thisname+'_'+ecoregname+'.sav'
            #print('\tLoading model: '+ filename)
            climmodeldict[thisname] = joblib.load(filename)
            # retrieve models
            modelinputs = modelinputsst_string(landtypeinput, thisname,
                                              ithisecoreg,
                                              ithisecoregend)
            #print('\tModel inputs: '+', '.join(modelinputs))
            # get the column(s) of SST
            if len(modelinputs)==1:
                tmpX = np.asarray(SSTgradient[modelinputs]).reshape(-1,1)
            else:
                tmpX = np.asarray(SSTgradient[modelinputs])
            # predict climate, choose timeframe 1984 to present
            tmppred = climmodeldict[thisname].predict(tmpX).flatten()[1:].tolist()
            #print('\tCoef: '+str(np.round(climmodeldict[thisname].coef_, decimals=3)))
            climpred.append(tmppred)
            climnames.append(thisname)
    
    if len(climy1names)>0:
        for thisname in climy1names:
            #print(thisname)
            filename = model_string + 'model_'+landname+'_'+thisname+'_'+ecoregname+'.sav'
            filename = filename.replace("y-1", "y0") # change to get model name
            tmpname = thisname.replace("y-1", "y0")
            #print('\tLoading model: '+ filename)
            climmodeldict[thisname] = joblib.load(filename)
            # retrieve models
            modelinputs = modelinputsst_string(landtypeinput, tmpname,
                                               ithisecoreg,
                                               ithisecoregend)
            #print('\tModel inputs: '+' '.join(modelinputs))
            # get the column(s) of SST
            if len(modelinputs)==1:
                tmpX = np.asarray(SSTgradient[modelinputs]).reshape(-1,1)
            else:
                tmpX = np.asarray(SSTgradient[modelinputs])
            # predict climate, choose timeframe 1983 to prior-year
            tmppred = climmodeldict[thisname].predict(tmpX).flatten()[:-1].tolist()
            #print('\tCoef: '+str(np.round(climmodeldict[thisname].coef_, decimals=3)))
            climpred.append(tmppred)
            climnames.append(thisname)
    
    # create pandas dataframe to feed into prediction, sort
    climpred_T = np.array(climpred).transpose()
    Xclimpred = pd.DataFrame(climpred_T, index=None, columns=climnames).reindex(climallnames, axis=1)
    
    climbyecoregfor.append(Xclimpred)
print('The shape of our final SST-OBS FOREST climates: '+str(np.shape(climbyecoregfor)))

allwestUS
ecoprov1
ecoprov2
ecoprov3
ecoprov4


ecoprov5


ecoprov6


ecoprov7
ecoprov8
ecoprov9
ecoprov10
ecoprov12
ecoprov13
ecoprov14
ecoprov15


ecoprov16
ecoprov17
ecoprov18
ecoprov19
ecoprov20


ecoprov21


The shape of our final SST-OBS FOREST climates: (21, 39, 72)


In [12]:
# do the same for NONFOREST
for ecoregname in dfnames:
    print(ecoregname)
    landname = 'non'
    landtypeinput = inputsnon
    landtypeiinput = iinputsnon

    ithisecoreg = landtypeinput.index('+++'+ecoregname)
    ithisecoregend = landtypeiinput[landtypeiinput.index(ithisecoreg)+1]-1
    # storage
    climmodeldict = {} # the loaded model
    climpred = [] # predicted climate variable values
    climnames = [] # name of climate variable
    
    # open models
    if len(climy0names)>0:
        for thisname in climy0names:
            #print(thisname)
            filename = model_string + 'model_'+landname+'_'+thisname+'_'+ecoregname+'.sav'
            #print('\tLoading model: '+ filename)
            climmodeldict[thisname] = joblib.load(filename)
            # retrieve models
            modelinputs = modelinputsst_string(landtypeinput, thisname,
                                              ithisecoreg,
                                              ithisecoregend)
            #print('\tModel inputs: '+', '.join(modelinputs))
            # get the column(s) of SST
            if len(modelinputs)==1:
                tmpX = np.asarray(SSTgradient[modelinputs]).reshape(-1,1)
            else:
                tmpX = np.asarray(SSTgradient[modelinputs])
            # predict climate, choose timeframe 1984 to present
            tmppred = climmodeldict[thisname].predict(tmpX).flatten()[1:].tolist()
            #print('\tCoef: '+str(np.round(climmodeldict[thisname].coef_, decimals=3)))
            climpred.append(tmppred)
            climnames.append(thisname)
    
    if len(climy1names)>0:
        for thisname in climy1names:
            #print(thisname)
            filename = model_string + 'model_'+landname+'_'+thisname+'_'+ecoregname+'.sav'
            filename = filename.replace("y-1", "y0") # change to get model name
            tmpname = thisname.replace("y-1", "y0")
            #print('\tLoading model: '+ filename)
            climmodeldict[thisname] = joblib.load(filename)
            # retrieve models
            modelinputs = modelinputsst_string(landtypeinput, tmpname,
                                               ithisecoreg,
                                               ithisecoregend)
            #print('\tModel inputs: '+' '.join(modelinputs))
            # get the column(s) of SST
            if len(modelinputs)==1:
                tmpX = np.asarray(SSTgradient[modelinputs]).reshape(-1,1)
            else:
                tmpX = np.asarray(SSTgradient[modelinputs])
            # predict climate, choose timeframe 1983 to prior-year
            tmppred = climmodeldict[thisname].predict(tmpX).flatten()[:-1].tolist()
            #print('\tCoef: '+str(np.round(climmodeldict[thisname].coef_, decimals=3)))
            climpred.append(tmppred)
            climnames.append(thisname)
    
    # create pandas dataframe to feed into prediction, sort
    climpred_T = np.array(climpred).transpose()
    Xclimpred = pd.DataFrame(climpred_T, index=None, columns=climnames).reindex(climallnames, axis=1)
    
    climbyecoregnon.append(Xclimpred)
print('The shape of our final SST-OBS NONFOREST climates: '+str(np.shape(climbyecoregnon)))

allwestUS
ecoprov1
ecoprov2
ecoprov3
ecoprov4
ecoprov5


ecoprov6
ecoprov7
ecoprov8
ecoprov9
ecoprov10


ecoprov12


ecoprov13


ecoprov14
ecoprov15
ecoprov16
ecoprov17
ecoprov18
ecoprov19
ecoprov20
ecoprov21


The shape of our final SST-OBS NONFOREST climates: (21, 39, 72)


# SST gradient is detrended, SST->climate

In [13]:
# iterate through each FOREST ecoregion, detrended
# calculate the climate as predicted by the SST gradient

SSTgradient = climind82_seasonsDT

# separate concurrent and previous-year variables
climy0names = [s for s in climallnames if "y0" in s]
climy1names = [s for s in climallnames if "y-1" in s]


for ecoregname in dfnames:
    print(ecoregname)
    landname = 'for'
    landtypeinput = inputsfor
    landtypeiinput = iinputsfor

    ithisecoreg = landtypeinput.index('+++'+ecoregname)
    ithisecoregend = landtypeiinput[landtypeiinput.index(ithisecoreg)+1]-1
    # storage
    climmodeldict = {} # the loaded model
    climpred = [] # predicted climate variable values
    climnames = [] # name of climate variable
    
    # open models
    if len(climy0names)>0:
        for thisname in climy0names:
            #print(thisname)
            filename = model_string + 'model_'+landname+'_'+thisname+'_'+ecoregname+'.sav'
            #print('\tLoading model: '+ filename)
            climmodeldict[thisname] = joblib.load(filename)
            # retrieve models
            modelinputs = modelinputsst_string(landtypeinput, thisname,
                                              ithisecoreg,
                                              ithisecoregend)
            #print('\tModel inputs: '+', '.join(modelinputs))
            # get the column(s) of SST
            if len(modelinputs)==1:
                tmpX = np.asarray(SSTgradient[modelinputs]).reshape(-1,1)
            else:
                tmpX = np.asarray(SSTgradient[modelinputs])
            # predict climate, choose timeframe 1984 to present
            tmppred = climmodeldict[thisname].predict(tmpX).flatten()[1:].tolist()
            #print('\tCoef: '+str(np.round(climmodeldict[thisname].coef_, decimals=3)))
            climpred.append(tmppred)
            climnames.append(thisname)
    
    if len(climy1names)>0:
        for thisname in climy1names:
            #print(thisname)
            filename = model_string + 'model_'+landname+'_'+thisname+'_'+ecoregname+'.sav'
            filename = filename.replace("y-1", "y0") # change to get model name
            tmpname = thisname.replace("y-1", "y0")
            #print('\tLoading model: '+ filename)
            climmodeldict[thisname] = joblib.load(filename)
            # retrieve models
            modelinputs = modelinputsst_string(landtypeinput, tmpname,
                                               ithisecoreg,
                                               ithisecoregend)
            #print('\tModel inputs: '+' '.join(modelinputs))
            # get the column(s) of SST
            if len(modelinputs)==1:
                tmpX = np.asarray(SSTgradient[modelinputs]).reshape(-1,1)
            else:
                tmpX = np.asarray(SSTgradient[modelinputs])
            # predict climate, choose timeframe 1983 to prior-year
            tmppred = climmodeldict[thisname].predict(tmpX).flatten()[:-1].tolist()
            #print('\tCoef: '+str(np.round(climmodeldict[thisname].coef_, decimals=3)))
            climpred.append(tmppred)
            climnames.append(thisname)
    
    # create pandas dataframe to feed into prediction, sort
    climpred_T = np.array(climpred).transpose()
    Xclimpred = pd.DataFrame(climpred_T, index=None, columns=climnames).reindex(climallnames, axis=1)
    
    climbyecoregforDT.append(Xclimpred)
print('The shape of our final SST-DT FOREST climates: '+str(np.shape(climbyecoregfor)))

allwestUS
ecoprov1
ecoprov2
ecoprov3


ecoprov4


ecoprov5


ecoprov6
ecoprov7
ecoprov8
ecoprov9
ecoprov10
ecoprov12
ecoprov13
ecoprov14


ecoprov15


ecoprov16
ecoprov17
ecoprov18
ecoprov19


ecoprov20


ecoprov21


The shape of our final SST-DT FOREST climates: (21, 39, 72)


In [14]:
# do the same for NONFOREST, SST is detrended
for ecoregname in dfnames:
    print(ecoregname)
    landname = 'non'
    landtypeinput = inputsnon
    landtypeiinput = iinputsnon

    ithisecoreg = landtypeinput.index('+++'+ecoregname)
    ithisecoregend = landtypeiinput[landtypeiinput.index(ithisecoreg)+1]-1
    # storage
    climmodeldict = {} # the loaded model
    climpred = [] # predicted climate variable values
    climnames = [] # name of climate variable
    
    # open models
    if len(climy0names)>0:
        for thisname in climy0names:
            #print(thisname)
            filename = model_string + 'model_'+landname+'_'+thisname+'_'+ecoregname+'.sav'
            #print('\tLoading model: '+ filename)
            climmodeldict[thisname] = joblib.load(filename)
            # retrieve models
            modelinputs = modelinputsst_string(landtypeinput, thisname,
                                              ithisecoreg,
                                              ithisecoregend)
            #print('\tModel inputs: '+', '.join(modelinputs))
            # get the column(s) of SST
            if len(modelinputs)==1:
                tmpX = np.asarray(SSTgradient[modelinputs]).reshape(-1,1)
            else:
                tmpX = np.asarray(SSTgradient[modelinputs])
            # predict climate, choose timeframe 1984 to present
            tmppred = climmodeldict[thisname].predict(tmpX).flatten()[1:].tolist()
            #print('\tCoef: '+str(np.round(climmodeldict[thisname].coef_, decimals=3)))
            climpred.append(tmppred)
            climnames.append(thisname)
    
    if len(climy1names)>0:
        for thisname in climy1names:
            #print(thisname)
            filename = model_string + 'model_'+landname+'_'+thisname+'_'+ecoregname+'.sav'
            filename = filename.replace("y-1", "y0") # change to get model name
            tmpname = thisname.replace("y-1", "y0")
            #print('\tLoading model: '+ filename)
            climmodeldict[thisname] = joblib.load(filename)
            # retrieve models
            modelinputs = modelinputsst_string(landtypeinput, tmpname,
                                               ithisecoreg,
                                               ithisecoregend)
            #print('\tModel inputs: '+' '.join(modelinputs))
            # get the column(s) of SST
            if len(modelinputs)==1:
                tmpX = np.asarray(SSTgradient[modelinputs]).reshape(-1,1)
            else:
                tmpX = np.asarray(SSTgradient[modelinputs])
            # predict climate, choose timeframe 1983 to prior-year
            tmppred = climmodeldict[thisname].predict(tmpX).flatten()[:-1].tolist()
            #print('\tCoef: '+str(np.round(climmodeldict[thisname].coef_, decimals=3)))
            climpred.append(tmppred)
            climnames.append(thisname)
    
    # create pandas dataframe to feed into prediction, sort
    climpred_T = np.array(climpred).transpose()
    Xclimpred = pd.DataFrame(climpred_T, index=None, columns=climnames).reindex(climallnames, axis=1)
    
    climbyecoregnonDT.append(Xclimpred)
print('The shape of our final SST-OBS NONFOREST climates: '+str(np.shape(climbyecoregnon)))

allwestUS
ecoprov1
ecoprov2
ecoprov3
ecoprov4
ecoprov5
ecoprov6


ecoprov7


ecoprov8
ecoprov9
ecoprov10
ecoprov12


ecoprov13


ecoprov14


ecoprov15


ecoprov16
ecoprov17
ecoprov18
ecoprov19
ecoprov20
ecoprov21
The shape of our final SST-OBS NONFOREST climates: (21, 39, 72)


# Calculate the difference between the models, get the detrended scenario of climate
After calculating the predicted climate from the observed SST trend and the detrended SST scenarios, we can get the difference, which means we get the **contribution from SST**

Then we take the difference between the observed climate and the contribution from SST to get the **detrended scenario of climate**

In [15]:
# to be exported:
climobsminusdiff_for = [] # observed minus model-difference
climobsminusdiff_non = []

for i, thisname in enumerate(dfnames):
    # get difference between models
    tmpdifffor = climbyecoregfor[i] - climbyecoregforDT[i]
    tmpdiffnon = climbyecoregnon[i] - climbyecoregnonDT[i]
    # get detrended-SST climate
    tmpobsDTfor = dfframesfor[thisname] - tmpdifffor
    tmpobsDTnon = dfframesnon[thisname] - tmpdiffnon
    # save
    climobsminusdiff_for.append(tmpobsDTfor)
    climobsminusdiff_non.append(tmpobsDTnon)
print(str(np.shape(climobsminusdiff_for)))
print(str(np.shape(climobsminusdiff_non)))

(21, 39, 72)
(21, 39, 72)


## Save files
Save in the same format of `Data_CreateModelData.ipynb` for the observed climate variables

In [16]:
# create pandas dfs
dfframesforDT = {}
dfframesnonDT = {}
for i in np.arange(len(province_num)):
    dfframesforDT[dfnames[i]] = pd.DataFrame(climobsminusdiff_for[i], columns=climallnames)
    dfframesnonDT[dfnames[i]] = pd.DataFrame(climobsminusdiff_non[i], columns=climallnames)

In [17]:
# export the climate data by ecoprovince
for i in np.arange(len(province_num)):
    filename = predict_string + 'climateDT_ecoprovinces_'
    dfframesforDT[dfnames[i]].to_csv(filename + dfnames[i]+'_for.txt')
    dfframesnonDT[dfnames[i]].to_csv(filename + dfnames[i]+'_non.txt')
    print('Exported for, non - '+filename + dfnames[i])

# print last time the data was updated
from datetime import datetime

# datetime object containing current date and time
now = datetime.now()
# dd/mm/YY H:M:S
dt_string = now.strftime("%d/%m/%Y %H:%M:%S")
print("Data last updated =", dt_string)

Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_allwestUS
Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov1
Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov2
Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov3
Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov4
Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov5
Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov6
Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov7
Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov8


Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov9
Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov10
Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov12
Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov13


Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov14
Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov15
Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov16
Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov17


Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov18
Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov19
Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov20
Exported for, non - predicted//patch125-155_nino3-34//climateDT_ecoprovinces_ecoprov21
Data last updated = 18/11/2025 10:52:46
